# Régler `setfit` contre le lexique : le second banc d'essai

Le premier banc a rendu son verdict — **non** pour les trois montages — et
[Trancher si l'échec du troisième témoin est celui de la famille ou du réglage](https://github.com/AmauryTISSOT/microservice_rgpd/issues/55)
a établi que cet échec est celui du **réglage**, pas de la famille. Ce notebook éprouve les trois
leviers que ce ticket a nommés.

- Carte : [Un troisième moyen de détection, non génératif, pour affiner le diagnostic](https://github.com/AmauryTISSOT/microservice_rgpd/issues/42)
- Ticket : [Éprouver les leviers de réglage de setfit contre le lexique](https://github.com/AmauryTISSOT/microservice_rgpd/issues/56)
- Le premier banc : [`troisieme-temoin.ipynb`](troisieme-temoin.ipynb), dont ce notebook **prolonge**
  la machinerie sans la remplacer.

## Le critère, déclaré avant toute mesure

> **`setfit` dépasse le lexique en accord exact — strictement plus de 94/120 — au pire des 5 germes.**

Trois précisions qui en font partie et **ne se renégocient pas après coup** :

1. **Au pire des 5 germes**, comme le premier banc.
2. **L'estimation ponctuelle fait foi, pas la borne de Wilson.** L'intervalle est rapporté ; il ne
   fait pas gate.
3. **Aucune configuration ne se juge sur une graine.** Le papier SetFit rapporte σ ≈ 5 points
   entre graines à petit régime, et Wilson vaut ±8,6 points à n = 120 : **toute amélioration
   inférieure à ~9 points est indistinguable du bruit**. L'écart à combler — 20 à 28 exemples,
   soit 17 à 23 points — est heureusement au-dessus de ce plancher.

## Les trois leviers, et ce qui est écarté

| levier | état avant ce banc |
|---|---|
| **A** — `body_learning_rate` | **jamais réglé** ; facteur 50 entre le défaut du papier (1e-3) et celui de la bibliothèque (2e-5), et le premier banc a pris celui de la bibliothèque sans le choisir |
| **B** — l'encodeur | `multilingual-e5-small` est **le plus faible de MTEB-French** en Classification (0,60) |
| **C** — le seuil par étiquette | `SEUIL = 0,5`, non réglé |

Écartés par le ticket, et **pas réinstruits ici** : la tête (la doc officielle maintient la
régression logistique, déjà réglée plus finement que l'espace HPO officiel), `multi_target_strategy`
(`OneVsRestClassifier` **est** déjà `one-vs-rest`), la calibration (`setfit` est déjà le mieux
calibré des trois, Brier 0,070), et l'extension du corpus (hors périmètre de la carte).

## Ce que ce notebook ne fait pas

Il ne rédige ni ADR ni spec, ne touche à aucun code de production, ne redéfinit pas le
`ReviewSignal` et n'étend pas le corpus. **La lecture de ces chiffres et le verdict appartiennent à
un ticket distinct**, comme [Lire les chiffres du notebook et prononcer le verdict](https://github.com/AmauryTISSOT/microservice_rgpd/issues/52)
l'a été pour le premier banc. Ce notebook produit des chiffres ; il n'en tire pas de conclusion.

## L'environnement d'exécution

Imprimé et versionné, pour la même raison que dans le premier banc : sans cette cellule, un écart
de chiffres entre deux machines serait indécidable ; avec elle, il est diagnosticable. La
reproductibilité visée reste celle de
[Emplacement, outillage et reproductibilité du notebook](https://github.com/AmauryTISSOT/microservice_rgpd/issues/49) —
**même machine + même lock ⇒ mêmes chiffres**, machine différente ⇒ **même verdict** seulement.

In [1]:
import json, logging, math, os, platform, random, sys, tempfile, time, warnings
from collections import Counter
from pathlib import Path

# Bruit d'import et d'entraînement, sans rapport avec ce qui est mesuré. Filtré ici pour que les
# sorties versionnées restent lisibles — jamais pour taire un avertissement du banc d'essai.
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=".*pin_memory.*")

import numpy as np
import psutil
import sklearn
import torch
import transformers
import sentence_transformers
import setfit

# `banc.py` est le voisin de ce notebook. Le chemin est cherché en remontant plutôt que codé en
# dur : le notebook est lancé tantôt depuis `exploration/`, tantôt depuis la racine du dépôt.
for _dossier in (Path.cwd(), *Path.cwd().parents):
    if (_dossier / "banc.py").is_file():
        sys.path.insert(0, str(_dossier)); break
    if (_dossier / "exploration" / "banc.py").is_file():
        sys.path.insert(0, str(_dossier / "exploration")); break
else:
    raise RuntimeError("banc.py introuvable en remontant depuis le répertoire courant")

print(f"python                {platform.python_version()}  ({platform.machine()})")
print(f"plateforme            {platform.platform()}")
print(f"cœurs logiques        {os.cpu_count()}   physiques {psutil.cpu_count(logical=False)}")
print(f"torch                 {torch.__version__}   fils {torch.get_num_threads()}")
print(f"scikit-learn          {sklearn.__version__}")
print(f"transformers          {transformers.__version__}")
print(f"sentence-transformers {sentence_transformers.__version__}")
print(f"setfit                {setfit.__version__}")
transformers.logging.set_verbosity_error()
logging.getLogger("setfit").setLevel(logging.ERROR)

# Règle héritée du premier banc, et appliquée telle quelle : le corps contrastif s'entraîne sur la
# carte, la latence continue de se mesurer sur CPU. On mesure un coût là où il serait payé — la
# durée d'entraînement est un coût de *banc*, la latence un coût de *service*.
APPAREIL = "cuda" if torch.cuda.is_available() else "cpu"
if APPAREIL == "cuda":
    carte = torch.cuda.get_device_properties(0)
    vram_libre, vram_totale = torch.cuda.mem_get_info()
    print(f"CUDA                  {torch.version.cuda}   {carte.name}")
    print(f"VRAM                  {vram_totale / 1024 ** 3:.1f} Gio dont "
          f"{vram_libre / 1024 ** 3:.1f} Gio libres")
    if vram_libre < 5 * 1024 ** 3:
        print("  ATTENTION : moins de 5 Gio libres — un serveur de modèles occupe-t-il la carte ?")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    print("CUDA                  indisponible — compter plusieurs heures par configuration")
print(f"corps contrastif sur  {APPAREIL}    |    latence mesurée sur  cpu")

python                3.13.5  (AMD64)
plateforme            Windows-11-10.0.26200-SP0
cœurs logiques        16   physiques 8
torch                 2.13.0+cu126   fils 8
scikit-learn          1.9.0
transformers          4.57.6
sentence-transformers 5.6.1
setfit                1.1.3
CUDA                  12.6   NVIDIA GeForce RTX 3070 Laptop GPU
VRAM                  8.0 Gio dont 7.0 Gio libres
corps contrastif sur  cuda    |    latence mesurée sur  cpu


## La machinerie partagée, et la preuve qu'elle n'a pas dérivé

Les chiffres de ce banc doivent être **comparables** à ceux du premier : mêmes plis, même tête,
même règle d'arbitrage, même intervalle. Recopier trois cents lignes de cellules dans un second
notebook aurait rendu cette identité invérifiable — une ligne qui dérive déplace les chiffres sans
que rien ne le signale. La machinerie est donc extraite dans [`banc.py`](banc.py), que ce notebook
importe.

Le premier notebook, lui, **n'est pas modifié** : il est livré, ses sorties portent le verdict de
[Lire les chiffres du notebook et prononcer le verdict](https://github.com/AmauryTISSOT/microservice_rgpd/issues/52),
et le rejouer pour un refactoring serait payer six heures pour ne rien apprendre.

L'équivalence est donc prouvée **sur la donnée versionnée, pas par la relecture** :
`verifier_equivalence()` re-dérive les plis des 5 germes et rejoue la règle d'arbitrage sur les
1800 prédictions de `temoin-predictions.jsonl`. Un seul pli ou une seule décision qui diffère, et
la cellule lève. C'est la discipline « recalculé, jamais recopié » que le premier banc appliquait à
son point de comparaison, retournée cette fois vers son propre code.

In [2]:
import banc
from banc import (GERMES, GRILLE_SEUILS, K_EXTERNE, N_EXEMPLES, PLIS, PREFIXE_E5, SEUIL, SLUGS,
                  Y, accord, decider, identifiants, textes, verite, wilson)

equivalence = banc.verifier_equivalence()
print(f"plis re-dérivés et confrontés à l'artefact : {equivalence['plis_verifies']}")
print(f"décisions rejouées et confrontées          : {equivalence['decisions_verifiees']}")
print(f"montages                                   : {equivalence['montages']}")
print(f"germes                                     : {equivalence['germes']}")
print()
print(f"corpus : {N_EXEMPLES} exemples, {len(banc.GROUPES)} groupes "
      f"({sum(1 for g in banc.GROUPES if len(g) > 1)} paires minimales)")
print("étiquettes :", dict(zip(SLUGS, Y.sum(axis=0))))
print()
print(f"LA BARRE — le lexique en accord exact, recalculé et non recopié : "
      f"{banc.EXACTITUDE_LEXIQUE}/{N_EXEMPLES}")
print(f"Le critère demande **strictement plus**, soit au moins {banc.EXACTITUDE_LEXIQUE + 1}/"
      f"{N_EXEMPLES}, au pire des {len(GERMES)} germes.")

plis re-dérivés et confrontés à l'artefact : 1800
décisions rejouées et confrontées          : 1800
montages                                   : ['e5-gele', 'setfit', 'tfidf']
germes                                     : [20180525, 20190523, 20200101, 20210704, 20221123]

corpus : 120 exemples, 112 groupes (8 paires minimales)
étiquettes : {'acces': np.int64(23), 'rectification': np.int64(14), 'effacement': np.int64(26), 'limitation': np.int64(14), 'portabilite': np.int64(13), 'opposition': np.int64(20), 'hors-perimetre': np.int64(30)}

LA BARRE — le lexique en accord exact, recalculé et non recopié : 94/120
Le critère demande **strictement plus**, soit au moins 95/120, au pire des 5 germes.


## Le point de départ — ce que le premier banc a mesuré

Relu depuis l'artefact versionné, jamais recopié d'un commentaire. C'est le chiffre que les trois
leviers doivent déplacer.

In [3]:
PREDICTIONS_PREMIER_BANC = banc.charger_predictions()

print(f"{'montage':<9} {'accord exact par germe':<34} {'pire':>5} {'médiane':>8} {'meilleur':>9}")
DEPART = {}
for montage in ("tfidf", "e5-gele", "setfit"):
    par_germe = banc.exactitudes_par_germe(PREDICTIONS_PREMIER_BANC, montage, GERMES)
    DEPART[montage] = par_germe
    valeurs = list(par_germe.values())
    print(f"{montage:<9} {str(valeurs):<34} {min(valeurs):>5} "
          f"{np.median(valeurs):>8.0f} {max(valeurs):>9}")
print(f"{'lexique':<9} {'— déterministe, un seul chiffre':<34} "
      f"{banc.EXACTITUDE_LEXIQUE:>5} {banc.EXACTITUDE_LEXIQUE:>8} {banc.EXACTITUDE_LEXIQUE:>9}")
print()
ecart = banc.EXACTITUDE_LEXIQUE - min(DEPART["setfit"].values())
print(f"L'écart à combler pour `setfit` : {ecart} exemples au pire germe "
      f"({ecart / N_EXEMPLES:.1%}), et il en faut {ecart + 1} pour franchir strictement.")

montage   accord exact par germe              pire  médiane  meilleur
tfidf     [49, 58, 53, 54, 45]                  45       53        58
e5-gele   [63, 67, 63, 65, 59]                  59       63        67
setfit    [74, 69, 69, 71, 66]                  66       69        74
lexique   — déterministe, un seul chiffre       94       94        94

L'écart à combler pour `setfit` : 28 exemples au pire germe (23.3%), et il en faut 29 pour franchir strictement.


## Levier C — le seuil par étiquette, à **coût nul**

Les 7 probabilités des 1800 prédictions étant versionnées, ce levier s'éprouve **sans réentraîner
quoi que ce soit**. Trois lectures, dans cet ordre :

1. **Le plafond oracle** — seuils choisis **en voyant toute la vérité**. C'est une borne supérieure
   inatteignable en pratique, et elle sert précisément à ça : si même elle ne suffit pas, inutile
   d'espérer d'un réglage honnête qu'il y arrive.
2. **Le réglage honnête** — pour chaque pli externe, les seuils sont choisis sur les **quatre
   autres plis** et appliqués au pli tenu à l'écart.
3. **Ce que le réglage retient** — les seuils effectivement choisis, étiquette par étiquette.

### Deux objectifs de réglage, et un seul est le bon

Un seuil se règle **contre quelque chose**, et ce quelque chose n'est pas neutre.

- **Objectif `f1`** — le F1 de chaque étiquette, réglé séparément. C'est la forme littérale de
  `TunedThresholdClassifierCV`, qui est binaire, appliquée par tête comme le ticket le demande.
- **Objectif `accord`** — l'accord exact des sept étiquettes **après la règle d'arbitrage**. C'est
  la métrique du critère, mot pour mot.

Ce ne sont pas deux façons de mesurer la même chose. La règle d'arbitrage **couple** les sept
têtes : le seuil de `hors-perimetre` décide si l'exclusivité s'applique, et le seuil d'une
étiquette de droit décide si `hors-perimetre` l'emporte sur elle. Un réglage par étiquette optimise
donc une quantité que le critère ne lit pas — et rien ne garantit que le gain se transporte. Les
deux sont mesurés ci-dessous, précisément pour que l'écart soit visible plutôt que supposé.

L'objectif `accord` n'est pas séparable, d'où une **montée par coordonnées** en deux passes,
partant de `0,5` partout et ne déplaçant un seuil que s'il gagne **strictement** — de sorte que le
montage non réglé garde la main sur toutes les égalités.

⚠️ **Écart au protocole, déclaré ici et non enfoui.** Le réglage « honnête » de ce levier n'est
pas parfaitement imbriqué : les probabilités des quatre plis de réglage viennent de modèles dont
l'apprentissage incluait le pli tenu à l'écart. La fuite est de second ordre — un scalaire par
étiquette, choisi sur une grille de 12 pas — mais elle est réelle, et elle joue **en faveur** du
réglage. Un réglage sans aucune fuite exigerait de réentraîner, ce qui contredirait le « coût nul »
qui fait tout l'intérêt de cette phase. Les phases suivantes, elles, règlent le seuil **dans les
plis internes**, donc sans cette fuite : la comparaison entre les deux chiffres mesure au passage
ce que la fuite vaut.

Deux garde-fous demandés par le ticket sont appliqués dans `banc.py` : **grille grossière**
(pas de 0,05, de 0,30 à 0,85) parce que sur 13-14 positifs un pli n'en voit que 2 ou 3 et qu'une
grille fine choisirait le bruit ; et **`0,5` gagne les égalités**, de sorte que le réglage ne bouge
que lorsqu'il gagne vraiment.

In [4]:
def probabilites_de(enregistrements, montage, germe):
    '''La matrice 120 x 7 des probabilités hors-pli, dans l'ordre du corpus.'''
    par_id = {r["id"]: r for r in enregistrements
              if r["montage"] == montage and r["germe"] == germe}
    assert len(par_id) == N_EXEMPLES, "Prédictions hors-pli incomplètes."
    return np.array([[par_id[i]["probabilites"][s] for s in SLUGS] for i in identifiants])


def accord_exact(P, seuils_par_exemple):
    '''L'accord exact sur les 120, la règle d'arbitrage appliquée avec un seuil par exemple.'''
    return sum(1 for i in range(N_EXEMPLES)
               if set(decider(P[i], seuils_par_exemple[i])["predit"]) == verite[i])


OBJECTIFS = ("accord", "f1")
SEUILS_ORACLE, SEUILS_HONNETES, PHASE_0 = {}, {}, {}

print(f"{'montage':<9} {'germe':>9} {'seuil 0,5':>10} | "
      f"{'oracle accord':>13} {'honnête accord':>15} | {'oracle f1':>10} {'honnête f1':>11}")
for montage in ("tfidf", "e5-gele", "setfit"):
    PHASE_0[montage] = {"base": {}}
    for objectif in OBJECTIFS:
        PHASE_0[montage][f"oracle-{objectif}"] = {}
        PHASE_0[montage][f"honnete-{objectif}"] = {}
    for germe in GERMES:
        P = probabilites_de(PREDICTIONS_PREMIER_BANC, montage, germe)
        plis = PLIS[germe]
        PHASE_0[montage]["base"][germe] = accord_exact(P, [None] * N_EXEMPLES)

        for objectif in OBJECTIFS:
            # Le plafond : les seuils voient la vérité des 120. Inatteignable, et c'est le propos.
            oracle_seuils = banc._regler_seuils(P, Y, objectif)
            PHASE_0[montage][f"oracle-{objectif}"][germe] = accord_exact(
                P, [oracle_seuils] * N_EXEMPLES)

            # Le réglage honnête : pour chaque pli, les seuils viennent des quatre autres.
            seuils_par_exemple, retenus = [None] * N_EXEMPLES, []
            for f in range(K_EXTERNE):
                autres = np.flatnonzero(plis != f)
                s = banc._regler_seuils(P[autres], Y[autres], objectif)
                retenus.append(s)
                for i in np.flatnonzero(plis == f):
                    seuils_par_exemple[int(i)] = s
            PHASE_0[montage][f"honnete-{objectif}"][germe] = accord_exact(P, seuils_par_exemple)
            SEUILS_ORACLE[(montage, germe, objectif)] = oracle_seuils
            SEUILS_HONNETES[(montage, germe, objectif)] = retenus

        r = PHASE_0[montage]
        print(f"{montage:<9} {germe:>9} {r['base'][germe]:>10} | "
              f"{r['oracle-accord'][germe]:>13} {r['honnete-accord'][germe]:>15} | "
              f"{r['oracle-f1'][germe]:>10} {r['honnete-f1'][germe]:>11}", flush=True)

montage       germe  seuil 0,5 | oracle accord  honnête accord |  oracle f1  honnête f1


tfidf      20180525         49 |            50              46 |         49          49


tfidf      20190523         58 |            58              49 |         58          48


tfidf      20200101         53 |            54              49 |         40          34


tfidf      20210704         54 |            55              52 |         54          54


...

setfit     20180525         74 |            85              75 |         82          75


setfit     20190523         69 |            81              71 |         79          73


setfit     20200101         69 |            78              66 |         70          65


setfit     20210704         71 |            74              66 |         72          64


setfit     20221123         66 |            76              65 |         74          68


### Ce que le levier C rapporte, au pire germe

Le critère se lit au **pire** des germes ; c'est donc la seule colonne qui décide.

In [5]:
print(f"la barre : {banc.EXACTITUDE_LEXIQUE}/{N_EXEMPLES} (le lexique) — il faut "
      f"{banc.EXACTITUDE_LEXIQUE + 1} au pire germe")
print()
print(f"{'montage':<9} {'0,5':>5} | {'oracle acc.':>11} {'honnête acc.':>12} {'gain':>5} | "
      f"{'oracle f1':>9} {'honnête f1':>10} {'gain':>5} | {'reste':>6}")
PIRES_PHASE_0 = {}
for montage in ("tfidf", "e5-gele", "setfit"):
    pires = {k: min(v.values()) for k, v in PHASE_0[montage].items()}
    PIRES_PHASE_0[montage] = pires
    reste = banc.EXACTITUDE_LEXIQUE + 1 - pires["honnete-accord"]
    print(f"{montage:<9} {pires['base']:>5} | {pires['oracle-accord']:>11} "
          f"{pires['honnete-accord']:>12} {pires['honnete-accord'] - pires['base']:>+5} | "
          f"{pires['oracle-f1']:>9} {pires['honnete-f1']:>10} "
          f"{pires['honnete-f1'] - pires['base']:>+5} | {reste:>6}")

print()
print("Les seuils honnêtes retenus pour `setfit`, objectif `accord`, par étiquette "
      f"(sur les {len(GERMES) * K_EXTERNE} plis) :")
tous = [s for germe in GERMES for s in SEUILS_HONNETES[("setfit", germe, "accord")]]
for l, slug in enumerate(SLUGS):
    colonne = [s[l] for s in tous]
    inchanges = sum(1 for v in colonne if v == SEUIL)
    print(f"  {slug:<15} médiane {np.median(colonne):.2f}   laissé à 0,5 : "
          f"{inchanges}/{len(colonne)}   "
          f"choix : {', '.join(f'{v}×{n}' for v, n in Counter(colonne).most_common(4))}")
print()
print("Renseignement directionnel : un seuil médian **au-dessus** de 0,5 signifie que le modèle")
print("sur-prédit l'étiquette, et un seuil en dessous qu'il la sous-prédit.")

la barre : 94/120 (le lexique) — il faut 95 au pire germe

montage     0,5 | oracle acc. honnête acc.  gain | oracle f1 honnête f1  gain |  reste
tfidf        45 |          47           44    -1 |        40         34   -11 |     51
e5-gele      59 |          63           48   -11 |        59         53    -6 |     47
setfit       66 |          74           65    -1 |        70         64    -2 |     30

Les seuils honnêtes retenus pour `setfit`, objectif `accord`, par étiquette (sur les 25 plis) :
  acces           médiane 0.65   laissé à 0,5 : 6/25   choix : 0.5×6, 0.7×5, 0.65×4, 0.3×3
  rectification   médiane 0.60   laissé à 0,5 : 7/25   choix : 0.5×7, 0.6×7, 0.7×4, 0.65×3
  effacement      médiane 0.55   laissé à 0,5 : 9/25   choix : 0.5×9, 0.55×6, 0.65×6, 0.4×3
  limitation      médiane 0.50   laissé à 0,5 : 20/25   choix : 0.5×20, 0.55×4, 0.65×1
  portabilite     médiane 0.60   laissé à 0,5 : 7/25   choix : 0.6×8, 0.5×7, 0.65×4, 0.75×4
  opposition      médiane 0.75   laissé à 0

### Ce que la phase 0 établit — et l'écart entre les deux objectifs

Le tableau ci-dessus est à lire dans cet ordre.

1. **Le plafond oracle de l'objectif `accord`** est la seule borne supérieure qui compte, puisque
   c'est la métrique du critère. Si elle laisse `setfit` loin du lexique, le levier C ne peut pas,
   à lui seul, combler l'écart — et un réglage honnête en récupérera moins encore.
2. **L'objectif `f1` fait moins bien que l'objectif `accord`** sur la métrique du critère. Ce n'est
   pas une surprise mais une conséquence : optimiser le F1 par étiquette optimise une quantité que
   le critère ne lit pas. C'est la trace chiffrée de ce que le couplage par la règle d'arbitrage
   coûte, et la raison pour laquelle tout ce qui suit règle sur `accord`.
3. **L'écart oracle → honnête** mesure ce que le réglage perd en cessant de tricher. Sur 13-14
   positifs répartis en 5 plis, c'est là que le bruit se paie.

⚠️ Un réglage **honnête inférieur au seuil non réglé** n'est pas un bug : la montée par
coordonnées ne peut pas dégrader le jeu **sur lequel elle règle**, mais rien ne l'empêche de
dégrader le pli tenu à l'écart. C'est la définition du sur-ajustement, et c'est exactement ce que
le ticket redoutait en écrivant qu'un seuil réglé sur ~2-3 positifs par pli « risque d'être du
bruit ».

### Seuil réglé **ou** `class_weight="balanced"` — la question que le ticket demande de trancher

Régler le seuil **et** pondérer les classes corrige deux fois le même déséquilibre. Le ticket
demande de choisir lequel garder, et de l'écrire. On regarde d'abord ce que la tête a réellement
retenu dans le premier banc, puis on tranche.

In [6]:
#: Un enregistrement par exemple de test : on repasse aux plis en dédupliquant sur (germe, pli).
def configurations_par_pli(enregistrements, montage):
    return {(r["germe"], r["pli"]): (r["reglage"]["C"], r["reglage"]["class_weight"])
            for r in enregistrements if r["montage"] == montage}


N_PLIS = len(GERMES) * K_EXTERNE
print(f"Configurations retenues par la tête dans le premier banc (sur {N_PLIS} plis par montage) :")
for montage in ("tfidf", "e5-gele", "setfit"):
    compte = Counter(configurations_par_pli(PREDICTIONS_PREMIER_BANC, montage).values())
    assert sum(compte.values()) == N_PLIS
    print(f"  {montage:<9} " + "   ".join(
        f"C={c} cw={w or 'None'} : {n}" for (c, w), n in sorted(compte.items(), key=lambda x: -x[1])))

equilibrees = sum(1 for _, w in configurations_par_pli(PREDICTIONS_PREMIER_BANC, "setfit").values()
                  if w == "balanced")
print()
print(f"Pour `setfit`, la tête a choisi `class_weight=\"balanced\"` sur {equilibrees} "
      f"des {N_PLIS} plis.")

Configurations retenues par la tête dans le premier banc (sur 25 plis par montage) :
  tfidf     C=0.1 cw=balanced : 12   C=1.0 cw=balanced : 7   C=10.0 cw=balanced : 6
  e5-gele   C=10.0 cw=balanced : 22   C=1.0 cw=balanced : 2   C=0.1 cw=balanced : 1
  setfit    C=10.0 cw=balanced : 22   C=10.0 cw=None : 3

Pour `setfit`, la tête a choisi `class_weight="balanced"` sur 22 des 25 plis.


**La décision, écrite ici et appliquée dans tout ce qui suit : quand le seuil est réglé, la grille
de tête est restreinte à `class_weight=None`.**

Le raisonnement, en trois temps.

1. Les deux mécanismes agissent au **même endroit** — la position de la frontière d'une tête
   `one-vs-rest` par rapport à la prévalence de son étiquette. `class_weight="balanced"` la déplace
   *avant* l'ajustement, en réécrivant la fonction de perte ; le seuil la déplace *après*, sur la
   sortie. Les composer ne double pas la correction dans un sens utile : elle est appliquée deux
   fois sur la même quantité, et le second réglage passe son temps à défaire le premier.
2. Entre les deux, **le seuil est le plus mesurable**. `balanced` est un facteur imposé par la
   prévalence, sans degré de liberté ; le seuil est choisi sur la donnée, et son choix est
   observable pli par pli — on peut donc lire *ce qu'il a retenu*, ce que `balanced` n'offre pas.
3. Le coût de la décision est **borné et connu** : `class_weight` reste dans la grille des
   configurations qui ne règlent pas le seuil, si bien que le banc mesure les deux voies plutôt
   que d'en supposer une meilleure.

Conséquence pratique : la grille de tête à seuil réglé est `GRILLE_TETE_SANS_POIDS`, trois
configurations au lieu de six. Le budget d'entraînement en profite, mais ce n'est pas la raison —
c'est un effet de bord agréable d'une décision prise sur le fond.

In [7]:
GRILLE_TETE_SANS_POIDS = [{"C": c, "class_weight": None} for c in (0.1, 1.0, 10.0)]
print("grille à seuil réglé   :", GRILLE_TETE_SANS_POIDS)
print("grille du premier banc :", banc.GRILLE_TETE)

grille à seuil réglé   : [{'C': 0.1, 'class_weight': None}, {'C': 1.0, 'class_weight': None}, {'C': 10.0, 'class_weight': None}]
grille du premier banc : [{'C': 0.1, 'class_weight': None}, {'C': 0.1, 'class_weight': 'balanced'}, {'C': 1.0, 'class_weight': None}, {'C': 1.0, 'class_weight': 'balanced'}, {'C': 10.0, 'class_weight': None}, {'C': 10.0, 'class_weight': 'balanced'}]


## Le budget de calcul — mesuré avant d'être dépensé

C'est la contrainte dimensionnante du ticket, et elle se mesure au lieu de s'estimer. Un corps
`e5-small` coûtait 72 s sur la carte au premier banc, soit 33 min pour les 25 corps d'un montage
complet. Une grille de 6 valeurs de `body_learning_rate` × 5 germes × 5 plis coûterait ~3 h, et un
encodeur de 568 M paramètres multiplierait encore la facture.

Cette cellule entraîne **un seul corps par encodeur candidat**, sur le pli 0 du premier germe, et
en relève la durée et la VRAM de crête. Les phases 1 et 2 sont dimensionnées sur ces chiffres — pas
sur une extrapolation.

Les quatre candidats et leurs pièges, vérifiés par
[Trancher si l'échec du troisième témoin est celui de la famille ou du réglage](https://github.com/AmauryTISSOT/microservice_rgpd/issues/55)
et **à ne pas redécouvrir** : `bge-m3` et `Solon` **ne veulent pas** le préfixe `"query: "` de la
famille E5 — le leur imposer dégraderait les vecteurs *sans aucun signal* ;
`jina-embeddings-v3` est en CC-BY-NC-4.0, donc disqualifié ; `gte-multilingual-base` exige
`trust_remote_code=True`, contraire à l'auto-hébergement strict de l'ADR-0001.

In [8]:
#: Les encodeurs candidats, **révisions épinglées** — sans quoi la reproductibilité serait creuse.
#: Le préfixe est une propriété du modèle et non un réglage : la famille E5 l'exige, `bge-m3` et
#: `Solon` le refusent.
ENCODEURS = {
    "e5-small":  {"nom": "intfloat/multilingual-e5-small",
                  "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
                  "prefixe": PREFIXE_E5, "parametres_m": 118, "mteb_fr_classification": 0.60},
    "e5-base":   {"nom": "intfloat/multilingual-e5-base",
                  "revision": "d128750597153bb5987e10b1c3493a34e5a4502a",
                  "prefixe": PREFIXE_E5, "parametres_m": 278, "mteb_fr_classification": 0.65},
    "bge-m3":    {"nom": "BAAI/bge-m3",
                  "revision": "5617a9f61b028005a4858fdac845db406aefb181",
                  "prefixe": "", "parametres_m": 568, "mteb_fr_classification": 0.69},
    "solon-large": {"nom": "OrdalieTech/Solon-embeddings-large-0.1",
                    "revision": "9f6465f6ea2f6d10c6294bc15d84edf87d47cdef",
                    "prefixe": "", "parametres_m": 560, "mteb_fr_classification": 0.69},
}

for cle, spec in ENCODEURS.items():
    print(f"{cle:<12} {spec['nom']:<45} {spec['parametres_m']:>4} M   "
          f"MTEB-fr classif. {spec['mteb_fr_classification']:.2f}   "
          f"préfixe {'«' + spec['prefixe'] + '»' if spec['prefixe'] else 'aucun'}")

e5-small     intfloat/multilingual-e5-small                 118 M   MTEB-fr classif. 0.60   préfixe «query: »
e5-base      intfloat/multilingual-e5-base                  278 M   MTEB-fr classif. 0.65   préfixe «query: »
bge-m3       BAAI/bge-m3                                    568 M   MTEB-fr classif. 0.69   préfixe aucun
solon-large  OrdalieTech/Solon-embeddings-large-0.1         560 M   MTEB-fr classif. 0.69   préfixe aucun


In [9]:
from datasets import Dataset
import datasets
from setfit import SetFitModel, Trainer, TrainingArguments

datasets.disable_progress_bars()

# SetFit écrit ses points de reprise dans `checkpoints/` du répertoire courant par défaut. Le banc
# ne reprend rien : les envoyer au temporaire système évite de salir le dépôt.
SORTIE_SETFIT = tempfile.mkdtemp(prefix="setfit-reglage-")


def entrainer_corps(encodeur, appr, germe, lr_corps, max_seq=None, iterations=None):
    '''Entraîne un corps contrastif sur le seul jeu d'apprentissage, puis plonge les 120 textes.

    Rend les plongements et le coût observé. Les germes sont replantés à chaque appel : le corps
    contrastif échantillonne ses paires, et c'est le seul aléa du montage 3 en dehors du découpage.
    '''
    spec = ENCODEURS[encodeur]
    random.seed(germe)
    np.random.seed(germe % 2 ** 32)
    torch.manual_seed(germe)
    if APPAREIL == "cuda":
        torch.cuda.manual_seed_all(germe)
        torch.cuda.reset_peak_memory_stats()

    modele = SetFitModel.from_pretrained(
        spec["nom"], revision=spec["revision"], multi_target_strategy="one-vs-rest",
        device=APPAREIL)
    # Nos textes font 102 caractères de médiane. Plafonner la longueur ne tronque donc rien et
    # évite qu'un encodeur à fenêtre de 8192 jetons (bge-m3) alloue des tampons pour du vide.
    if max_seq is not None:
        modele.model_body.max_seq_length = max_seq

    X = [spec["prefixe"] + textes[int(i)] for i in appr]
    Y_ = [[int(v) for v in Y[int(i)]] for i in appr]
    arguments = TrainingArguments(
        batch_size=banc.SETFIT_LOT, num_epochs=banc.SETFIT_EPOQUES,
        num_iterations=iterations or banc.SETFIT_ITERATIONS, seed=germe,
        body_learning_rate=lr_corps, output_dir=SORTIE_SETFIT, report_to="none")

    debut = time.perf_counter()
    Trainer(model=modele, args=arguments,
            train_dataset=Dataset.from_dict({"text": X, "label": Y_})
            ).train_embeddings(x_train=X, y_train=Y_, args=arguments)
    # Les lancements CUDA sont asynchrones : sans cette barrière, le chronomètre mesurerait le
    # temps mis à *soumettre* le travail, pas celui mis à le faire.
    if APPAREIL == "cuda":
        torch.cuda.synchronize()
    duree = time.perf_counter() - debut
    vram = torch.cuda.max_memory_allocated() if APPAREIL == "cuda" else 0

    plongements = modele.model_body.encode(
        [spec["prefixe"] + t for t in textes], normalize_embeddings=True, batch_size=16)
    dimension = plongements.shape[1]
    del modele
    # Des corps chargés à la file remplissent la carte si on ne rend jamais rien : le cache
    # d'allocation de `torch` garde la mémoire libérée jusqu'à ce qu'on la lui redemande.
    if APPAREIL == "cuda":
        torch.cuda.empty_cache()
    return plongements, {"duree_s": duree, "vram_gio": vram / 1024 ** 3, "dimension": dimension}


#: La longueur de séquence plafonnée, commune à tous les candidats pour que la comparaison porte
#: sur l'encodeur et non sur sa fenêtre. Nos textes font 102 caractères de médiane : rien n'est
#: tronqué, et un encodeur à fenêtre de 8192 jetons n'alloue pas de tampons pour du vide.
MAX_SEQ = 128

#: L'étalonnage est une **sonde**, à un dixième des itérations, et non un entraînement complet.
#: La raison est mesurée et non supposée : un corps `bge-m3` complet a coûté plus de 55 min sur
#: cette carte, si bien qu'étalonner grandeur nature aurait coûté plus cher que ce que
#: l'étalonnage sert à décider. Le coût de l'entraînement contrastif est linéaire en
#: `num_iterations` — c'est le nombre de paires échantillonnées —, mais cette linéarité est
#: **vérifiée sur `e5-small`** juste en dessous plutôt que tenue pour acquise.
SONDE_ITERATIONS = 2

In [10]:
_appr_etalon = np.flatnonzero(PLIS[GERMES[0]] != 0)

# La vérification de linéarité : le même corps, à 2 puis à 20 itérations. Si le rapport observé
# s'écarte franchement de 10, l'extrapolation qui suit ne vaut rien et il faut le savoir ici.
_, sonde_petite = entrainer_corps("e5-small", _appr_etalon, GERMES[0], banc.SETFIT_LR_CORPS,
                                  MAX_SEQ, iterations=SONDE_ITERATIONS)
_, sonde_pleine = entrainer_corps("e5-small", _appr_etalon, GERMES[0], banc.SETFIT_LR_CORPS,
                                  MAX_SEQ, iterations=banc.SETFIT_ITERATIONS)
FACTEUR_ATTENDU = banc.SETFIT_ITERATIONS / SONDE_ITERATIONS
FACTEUR_MESURE = sonde_pleine["duree_s"] / sonde_petite["duree_s"]

print(f"e5-small à {SONDE_ITERATIONS:>2} itérations : {sonde_petite['duree_s']:>6.1f} s")
print(f"e5-small à {banc.SETFIT_ITERATIONS:>2} itérations : {sonde_pleine['duree_s']:>6.1f} s")
print(f"facteur attendu {FACTEUR_ATTENDU:.1f}  |  facteur mesuré {FACTEUR_MESURE:.1f}")
LINEARITE_VERIFIEE = abs(FACTEUR_MESURE - FACTEUR_ATTENDU) <= 0.3 * FACTEUR_ATTENDU
print("La sonde s'extrapole :", "oui" if LINEARITE_VERIFIEE else
      "NON — l'extrapolation ci-dessous est à lire comme un ordre de grandeur, pas un chiffre")

{'embedding_loss': 0.3981, 'grad_norm': 0.5505594611167908, 'learning_rate': 0.0, 'epoch': 0.04}


{'train_runtime': 11.8858, 'train_samples_per_second': 32.644, 'train_steps_per_second': 2.103, 'train_loss': 0.3017550051212311, 'epoch': 1.0}


{'embedding_loss': 0.2034, 'grad_norm': 0.29571324586868286, 'learning_rate': 0.0, 'epoch': 0.00411522633744856}


{'embedding_loss': 0.2798, 'grad_norm': 1.3635255098342896, 'learning_rate': 1.779816513761468e-05, 'epoch': 0.205761316872428}


{'embedding_loss': 0.1809, 'grad_norm': 1.0379664897918701, 'learning_rate': 1.3211009174311929e-05, 'epoch': 0.411522633744856}


{'embedding_loss': 0.133, 'grad_norm': 1.5287922620773315, 'learning_rate': 8.623853211009175e-06, 'epoch': 0.6172839506172839}


{'embedding_loss': 0.0917, 'grad_norm': 1.4024015665054321, 'learning_rate': 4.036697247706423e-06, 'epoch': 0.823045267489712}


{'train_runtime': 65.3393, 'train_samples_per_second': 59.382, 'train_steps_per_second': 3.719, 'train_loss': 0.1573332524839252, 'epoch': 1.0}


e5-small à  2 itérations :   12.8 s
e5-small à 20 itérations :   66.1 s
facteur attendu 10.0  |  facteur mesuré 5.2
La sonde s'extrapole : NON — l'extrapolation ci-dessous est à lire comme un ordre de grandeur, pas un chiffre


In [11]:
ETALONNAGE = {"e5-small": sonde_pleine | {"sonde": False}}
for cle in ("e5-base", "bge-m3", "solon-large"):
    try:
        _, cout = entrainer_corps(cle, _appr_etalon, GERMES[0], banc.SETFIT_LR_CORPS,
                                  MAX_SEQ, iterations=SONDE_ITERATIONS)
        ETALONNAGE[cle] = {"duree_s": cout["duree_s"] * FACTEUR_MESURE,
                           "duree_sonde_s": cout["duree_s"], "vram_gio": cout["vram_gio"],
                           "dimension": cout["dimension"], "sonde": True}
        print(f"{cle:<12} sonde {cout['duree_s']:>6.1f} s → corps complet estimé "
              f"{ETALONNAGE[cle]['duree_s']:>7.1f} s   VRAM {cout['vram_gio']:>5.2f} Gio   "
              f"dimension {cout['dimension']}", flush=True)
    except Exception as erreur:                      # noqa: BLE001
        # Un candidat qui ne tient pas sur la carte est un **résultat** et non un accident :
        # l'ADR-0001 rappelle que le GPU est déjà pris par `qwen3:8b`. On le consigne et on passe.
        ETALONNAGE[cle] = {"echec": f"{type(erreur).__name__}: {erreur}", "sonde": True}
        print(f"{cle:<12} ÉCHEC — {type(erreur).__name__}: {erreur}", flush=True)
        if APPAREIL == "cuda":
            torch.cuda.empty_cache()

{'embedding_loss': 0.3807, 'grad_norm': 1.6496294736862183, 'learning_rate': 0.0, 'epoch': 0.04}


{'train_runtime': 21.6073, 'train_samples_per_second': 17.957, 'train_steps_per_second': 1.157, 'train_loss': 0.26766037225723266, 'epoch': 1.0}


e5-base      sonde   22.3 s → corps complet estimé   115.6 s   VRAM  5.40 Gio   dimension 768


{'embedding_loss': 0.2313, 'grad_norm': 1.527613878250122, 'learning_rate': 0.0, 'epoch': 0.04}


{'train_runtime': 277.2565, 'train_samples_per_second': 1.399, 'train_steps_per_second': 0.09, 'train_loss': 0.19889636874198913, 'epoch': 1.0}


bge-m3       sonde  278.6 s → corps complet estimé  1440.6 s   VRAM 11.67 Gio   dimension 1024


{'embedding_loss': 0.2737, 'grad_norm': 3.1871907711029053, 'learning_rate': 0.0, 'epoch': 0.04}


{'train_runtime': 281.0159, 'train_samples_per_second': 1.381, 'train_steps_per_second': 0.089, 'train_loss': 0.196256206035614, 'epoch': 1.0}


solon-large  sonde  283.6 s → corps complet estimé  1466.3 s   VRAM 11.55 Gio   dimension 1024


In [12]:
print("Coût d'un montage complet, extrapolé de la sonde ci-dessus :")
print(f"{'encodeur':<12} {'1 corps':>9} {'5 plis (1 germe)':>18} {'25 plis (5 germes)':>20} "
      f"{'VRAM':>8} {'source':>9}")
for cle, cout in ETALONNAGE.items():
    if "echec" in cout:
        print(f"{cle:<12} indisponible — {cout['echec'][:70]}")
        continue
    un = cout["duree_s"]
    print(f"{cle:<12} {un:>7.0f} s {un * K_EXTERNE / 60:>16.1f} min "
          f"{un * K_EXTERNE * len(GERMES) / 60:>18.1f} min {cout['vram_gio']:>7.2f} G "
          f"{'sonde ×' + f'{FACTEUR_MESURE:.1f}' if cout['sonde'] else 'mesuré':>9}")
print()
print("C'est ce tableau, et lui seul, qui dimensionne les phases 1 et 2 ci-dessous.")

Coût d'un montage complet, extrapolé de la sonde ci-dessus :
encodeur       1 corps   5 plis (1 germe)   25 plis (5 germes)     VRAM    source
e5-small          66 s              5.5 min               27.5 min    2.53 G    mesuré
e5-base          116 s              9.6 min               48.1 min    5.40 G sonde ×5.2
bge-m3          1441 s            120.1 min              600.3 min   11.67 G sonde ×5.2
solon-large     1466 s            122.2 min              610.9 min   11.55 G sonde ×5.2

C'est ce tableau, et lui seul, qui dimensionne les phases 1 et 2 ci-dessous.


## Le banc étagé — ce qui est éprouvé, et ce qui est abandonné

Le tableau de la cellule précédente commande, avec une réserve importante sur la façon de le lire.

⚠️ **La sonde majore, et il faut le savoir avant de s'en servir.** Elle mesure un corps à 2
itérations et multiplie par le facteur observé sur `e5-small`. Or une partie du coût d'un corps ne
dépend **pas** du nombre d'itérations — charger le modèle, encoder les 120 textes à la fin. À 2
itérations, ces frais fixes pèsent lourd dans le total, et les multiplier par ~8 les compte huit
fois. La sonde rend donc une **borne supérieure**, d'autant plus large que le modèle est gros à
charger. Elle sert à écarter ce qui est manifestement hors d'atteinte, jamais à décider finement :
**l'admission en phase 2 se décide sur les durées réellement mesurées en phase 1**, pas sur elle.

⚠️ **Les deux gros encodeurs ne tiennent pas sur la carte, et c'est la cause de leur coût.** Ils
demandent ~11,6 Gio de crête là où la carte en offre 8 : le pilote déborde en mémoire hôte, et
l'entraînement s'effondre d'un facteur bien supérieur à ce que le rapport de paramètres laissait
attendre. Ce n'est pas un mauvais réglage de lot, et la marge est telle qu'aucune imprécision de la
sonde ne les ramènerait dans le budget. C'est la contrainte de l'ADR-0001 (« le GPU est déjà
pris ») rencontrée pour de bon — et sur une carte que **rien d'autre n'occupait** au moment de la
mesure. Servi, `qwen3:8b` en prendrait la moitié.

**Configurations abandonnées, déclarées et non tues** — le ticket l'exige, et une troncature
silencieuse se lirait comme une couverture exhaustive :

- `bge-m3` et `solon-large` : **écartés sur le budget**, pas sur leur mérite. Leur score MTEB-French
  en Classification (0,69 contre 0,60) reste le meilleur du lot ; ce banc ne dit pas qu'ils ne
  marcheraient pas, il dit qu'on ne peut pas le savoir ici. C'est un renseignement pour la carte,
  pas un verdict sur les modèles.

### Le partage entre les phases

Le ticket le laisse à la session qui exécute, sous une seule règle : déclarer le budget consommé et
les configurations abandonnées. Voici le partage retenu.

- **Phase 1 — dégrossissage.** Levier A sur `e5-small` : cinq valeurs de `body_learning_rate`
  couvrant la plage HPO officielle (`1e-6` → `1e-3` en échelle logarithmique), sur **deux germes**
  choisis pour encadrer — le meilleur et le pire du premier banc. Puis levier B : `e5-base` au
  meilleur `body_learning_rate`, sur **un germe**. Toute conclusion tirée ici est **provisoire**.
- **Phase 2 — confirmation.** Les finalistes seulement, sur les **5 germes**, critère appliqué sans
  retouche. Un encodeur y est admis si son coût **mesuré** en phase 1 le permet.

Les corps entraînés sont mis en cache sur disque, indexés par (encodeur, taux, germe, pli). Ce
n'est pas une commodité d'exécution : c'est ce qui permet à la phase 2 de ne repayer que les germes
que la phase 1 n'a pas déjà couverts. Le cache est une fonction pure de sa clé — le vider et
relancer redonne les mêmes chiffres, au prix du temps.

In [13]:
#: La plage HPO officielle de `setfit` est `1e-6` → `1e-3` en échelle logarithmique. Cinq points
#: y sont pris, dont les deux valeurs qui font la controverse : **2e-5**, le défaut de la
#: bibliothèque que le premier banc a pris sans le choisir, et **1e-3**, celui du papier.
GRILLE_LR_CORPS = [1e-6, 5e-6, 2e-5, 1e-4, 1e-3]

#: Deux germes qui **encadrent** : celui où `setfit` a fait le mieux au premier banc et celui où il
#: a fait le pire. Un dégrossissage sur deux germes voisins aurait tout le confort d'un résultat
#: stable et aucune de sa valeur.
_par_germe_setfit = DEPART["setfit"]
GERME_MEILLEUR = max(_par_germe_setfit, key=_par_germe_setfit.get)
GERME_PIRE = min(_par_germe_setfit, key=_par_germe_setfit.get)
GERMES_PHASE_1 = [GERME_PIRE, GERME_MEILLEUR]

CACHE_CORPS = Path(banc.RACINE) / "exploration" / ".cache-corps"
CACHE_CORPS.mkdir(exist_ok=True)

BUDGET = {"corps_entraines": 0, "secondes_gpu": 0.0, "corps_relus_du_cache": 0}
#: Les durées **réellement observées**, par encodeur. C'est cette table, et non la sonde, qui
#: décide de l'admission en phase 2.
DUREES_MESUREES = {}

#: Le plafond de budget de la phase 2, déclaré ici et non ajusté après coup : un encodeur y entre
#: si ses 25 corps tiennent dans une heure et demie de carte.
BUDGET_PHASE_2_S = 1.5 * 3600


def plongements(encodeur, lr_corps, germe, pli, appr):
    '''Les plongements des 120 textes par un corps entraîné sur le seul pli d'apprentissage.

    Le **coût** du corps est mis en cache avec ses plongements. Sans cela, une exécution servie
    par le cache annoncerait un budget de zéro et le banc mentirait sur ce qu'il a coûté — alors
    que déclarer ce budget est précisément ce que le ticket demande.
    '''
    cle = f"{encodeur}_lr{lr_corps:g}_g{germe}_p{pli}"
    chemin, chemin_cout = CACHE_CORPS / f"{cle}.npy", CACHE_CORPS / f"{cle}.json"
    if chemin.is_file() and chemin_cout.is_file():
        cout = json.loads(chemin_cout.read_text(encoding="utf-8"))
        BUDGET["corps_relus_du_cache"] += 1
        BUDGET["secondes_gpu"] += cout["duree_s"]
        DUREES_MESUREES.setdefault(encodeur, []).append(cout["duree_s"])
        return np.load(chemin)
    P, cout = entrainer_corps(encodeur, appr, germe, lr_corps, MAX_SEQ)
    BUDGET["corps_entraines"] += 1
    BUDGET["secondes_gpu"] += cout["duree_s"]
    DUREES_MESUREES.setdefault(encodeur, []).append(cout["duree_s"])
    np.save(chemin, P)
    chemin_cout.write_text(json.dumps(cout), encoding="utf-8")
    print(f"    corps {encodeur} lr={lr_corps:g} germe {germe} pli {pli} : "
          f"{cout['duree_s'] / 60:.1f} min   (budget cumulé "
          f"{BUDGET['secondes_gpu'] / 60:.0f} min sur {BUDGET['corps_entraines']} corps)",
          flush=True)
    return P


def montage_setfit(nom, encodeur, lr_corps, germes):
    '''Un montage complet : corps réentraîné dans **chaque pli externe**, seuil réglé en interne.'''
    def representation(appr, germe, pli):
        return plongements(encodeur, lr_corps, germe, pli, appr)

    return banc.mesurer(nom, representation, germes=germes, journal=True,
                        grille=GRILLE_TETE_SANS_POIDS, regler_seuils=True,
                        objectif_seuil="accord",
                        extra={"encodeur": encodeur, "body_learning_rate": lr_corps})


print(f"grille de `body_learning_rate` : {GRILLE_LR_CORPS}")
print(f"germes de la phase 1 : {GERMES_PHASE_1}  "
      f"(pire = {GERME_PIRE} à {_par_germe_setfit[GERME_PIRE]}/120, "
      f"meilleur = {GERME_MEILLEUR} à {_par_germe_setfit[GERME_MEILLEUR]}/120)")
print(f"cache des corps : {CACHE_CORPS.relative_to(banc.RACINE)}")
print(f"coût prévu de la phase 1, levier A : "
      f"{len(GRILLE_LR_CORPS) * len(GERMES_PHASE_1) * K_EXTERNE * 117 / 60:.0f} min")

grille de `body_learning_rate` : [1e-06, 5e-06, 2e-05, 0.0001, 0.001]
germes de la phase 1 : [20221123, 20180525]  (pire = 20221123 à 66/120, meilleur = 20180525 à 74/120)
cache des corps : exploration\.cache-corps
coût prévu de la phase 1, levier A : 98 min


### Phase 1, levier A — `body_learning_rate` sur `e5-small`

Le levier le mieux sourcé et le seul totalement inexploré. Deux germes, cinq taux.

In [14]:
PHASE_1 = {}
_debut_phase_1 = time.perf_counter()

for lr in GRILLE_LR_CORPS:
    nom = f"e5-small@{lr:g}"
    print(f"\n───── {nom}", flush=True)
    PHASE_1[nom] = montage_setfit(nom, "e5-small", lr, GERMES_PHASE_1)

DUREE_PHASE_1_A = time.perf_counter() - _debut_phase_1
print(f"\nphase 1, levier A : {DUREE_PHASE_1_A / 60:.1f} min")


───── e5-small@1e-06


  e5-small@1e-06 germe 20221123 pli 0 : {'C': 10.0, 'class_weight': None} seuils (0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3) macro-F1 interne 0.199 — 2.7 s


  e5-small@1e-06 germe 20221123 pli 1 : {'C': 10.0, 'class_weight': None} seuils (0.35, 0.3, 0.3, 0.35, 0.3, 0.35, 0.5) macro-F1 interne 0.289 — 2.3 s


  e5-small@1e-06 germe 20221123 pli 2 : {'C': 10.0, 'class_weight': None} seuils (0.4, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3) macro-F1 interne 0.220 — 2.1 s


  e5-small@1e-06 germe 20221123 pli 3 : {'C': 10.0, 'class_weight': None} seuils (0.35, 0.3, 0.3, 0.3, 0.35, 0.35, 0.5) macro-F1 interne 0.190 — 2.2 s


...

  e5-small@0.001 germe 20180525 pli 1 : {'C': 0.1, 'class_weight': None} seuils (0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5) macro-F1 interne 0.058 — 1.6 s


  e5-small@0.001 germe 20180525 pli 2 : {'C': 0.1, 'class_weight': None} seuils (0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5) macro-F1 interne 0.058 — 1.6 s


  e5-small@0.001 germe 20180525 pli 3 : {'C': 0.1, 'class_weight': None} seuils (0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5) macro-F1 interne 0.057 — 1.7 s


  e5-small@0.001 germe 20180525 pli 4 : {'C': 0.1, 'class_weight': None} seuils (0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5) macro-F1 interne 0.057 — 1.7 s



phase 1, levier A : 1.6 min


In [15]:
print(f"la barre : {banc.EXACTITUDE_LEXIQUE}/{N_EXEMPLES}")
print(f"le premier banc, sur ces deux germes : "
      f"{[DEPART['setfit'][g] for g in GERMES_PHASE_1]} (lr=2e-5, seuil 0,5, "
      f"grille de tête complète)")
print()
print(f"{'montage':<16} {'accord exact par germe':<24} {'pire':>5} {'écart au 1er banc':>18}")
CLASSEMENT_A = []
for nom, enregistrements in PHASE_1.items():
    par_germe = banc.exactitudes_par_germe(enregistrements, nom, GERMES_PHASE_1)
    valeurs = [par_germe[g] for g in GERMES_PHASE_1]
    depart = [DEPART["setfit"][g] for g in GERMES_PHASE_1]
    CLASSEMENT_A.append((min(valeurs), nom, par_germe))
    print(f"{nom:<16} {str(valeurs):<24} {min(valeurs):>5} "
          f"{str([v - d for v, d in zip(valeurs, depart)]):>18}")

CLASSEMENT_A.sort(reverse=True)
MEILLEUR_LR = float(CLASSEMENT_A[0][1].split("@")[1])
print()
print(f"Le meilleur taux sur ce dégrossissage : {MEILLEUR_LR:g} "
      f"(pire germe {CLASSEMENT_A[0][0]}/{N_EXEMPLES})")
print("⚠️ Provisoire : deux germes ne décident rien. Le critère se lit sur cinq, en phase 2.")
print(f"⚠️ Plancher de bruit : toute différence inférieure à ~9 exemples entre deux lignes de ce")
print(f"   tableau est indistinguable du bruit (σ ≈ 5 points entre graines, Wilson ±8,6 points).")

la barre : 94/120
le premier banc, sur ces deux germes : [66, 74] (lr=2e-5, seuil 0,5, grille de tête complète)

montage          accord exact par germe    pire  écart au 1er banc
e5-small@1e-06   [58, 64]                    58          [-8, -10]
e5-small@5e-06   [64, 68]                    64           [-2, -6]
e5-small@2e-05   [70, 74]                    70             [4, 0]
e5-small@0.0001  [66, 73]                    66            [0, -1]
e5-small@0.001   [30, 30]                    30         [-36, -44]

Le meilleur taux sur ce dégrossissage : 2e-05 (pire germe 70/120)
⚠️ Provisoire : deux germes ne décident rien. Le critère se lit sur cinq, en phase 2.
⚠️ Plancher de bruit : toute différence inférieure à ~9 exemples entre deux lignes de ce
   tableau est indistinguable du bruit (σ ≈ 5 points entre graines, Wilson ±8,6 points).


### Phase 1, levier B — l'encodeur

`e5-base` au meilleur taux du levier A, sur **un seul germe** : c'est tout ce que le budget permet
(~85 min pour cinq plis). Il ne peut donc pas prétendre au verdict, et sa mesure est un
renseignement directionnel sur le levier B.

In [16]:
_debut_phase_1_b = time.perf_counter()
GERMES_PHASE_1_B = [GERME_PIRE]
nom_b = f"e5-base@{MEILLEUR_LR:g}"
print(f"───── {nom_b} sur le seul germe {GERME_PIRE}", flush=True)
PHASE_1[nom_b] = montage_setfit(nom_b, "e5-base", MEILLEUR_LR, GERMES_PHASE_1_B)
DUREE_PHASE_1_B = time.perf_counter() - _debut_phase_1_b

reference = banc.exactitudes_par_germe(
    PHASE_1[f"e5-small@{MEILLEUR_LR:g}"], f"e5-small@{MEILLEUR_LR:g}", GERMES_PHASE_1_B)
mesure_b = banc.exactitudes_par_germe(PHASE_1[nom_b], nom_b, GERMES_PHASE_1_B)
print(f"\nphase 1, levier B : {DUREE_PHASE_1_B / 60:.1f} min")
print(f"  e5-small@{MEILLEUR_LR:g} au germe {GERME_PIRE} : {reference[GERME_PIRE]}/{N_EXEMPLES}")
print(f"  e5-base@{MEILLEUR_LR:g}  au germe {GERME_PIRE} : {mesure_b[GERME_PIRE]}/{N_EXEMPLES}")
print(f"  écart : {mesure_b[GERME_PIRE] - reference[GERME_PIRE]:+d} exemples, à comparer aux "
      f"+5 points que MTEB-French laissait espérer")
print("⚠️ Un germe. Sous le plancher de bruit, cet écart ne soutient aucune conclusion à lui seul.")

───── e5-base@2e-05 sur le seul germe 20221123


  e5-base@2e-05 germe 20221123 pli 0 : {'C': 10.0, 'class_weight': None} seuils (0.4, 0.3, 0.5, 0.3, 0.3, 0.3, 0.5) macro-F1 interne 0.942 — 1.8 s


{'embedding_loss': 0.2041, 'grad_norm': 0.6940945386886597, 'learning_rate': 0.0, 'epoch': 0.004201680672268907}


{'embedding_loss': 0.2504, 'grad_norm': 2.8876521587371826, 'learning_rate': 1.766355140186916e-05, 'epoch': 0.21008403361344538}


{'embedding_loss': 0.12, 'grad_norm': 1.3543287515640259, 'learning_rate': 1.2990654205607478e-05, 'epoch': 0.42016806722689076}


...

{'embedding_loss': 0.0548, 'grad_norm': 1.7120051383972168, 'learning_rate': 4.181818181818182e-06, 'epoch': 0.8163265306122449}


{'train_runtime': 76.6617, 'train_samples_per_second': 51.134, 'train_steps_per_second': 3.196, 'train_loss': 0.1136476559298379, 'epoch': 1.0}


    corps e5-base lr=2e-05 germe 20221123 pli 4 : 1.3 min   (budget cumulé 62 min sur 4 corps)


  e5-base@2e-05 germe 20221123 pli 4 : {'C': 10.0, 'class_weight': None} seuils (0.5, 0.3, 0.5, 0.3, 0.3, 0.35, 0.5) macro-F1 interne 0.968 — 85.9 s



phase 1, levier B : 6.3 min
  e5-small@2e-05 au germe 20221123 : 70/120
  e5-base@2e-05  au germe 20221123 : 77/120
  écart : +7 exemples, à comparer aux +5 points que MTEB-French laissait espérer
⚠️ Un germe. Sous le plancher de bruit, cet écart ne soutient aucune conclusion à lui seul.


### La sonde confrontée à la mesure

La phase 1 a réellement entraîné des corps des deux encodeurs. On peut donc confronter la sonde à
ce qu'elle prétendait annoncer — et c'est cette mesure, pas elle, qui décide de la phase 2.

In [17]:
print(f"{'encodeur':<12} {'sonde ×' + f'{FACTEUR_MESURE:.1f}':>14} {'mesuré (médiane)':>18} "
      f"{'écart':>8} {'25 corps':>10} {'phase 2':>9}")
ADMIS_PHASE_2 = {}
for cle in ("e5-small", "e5-base"):
    annonce = ETALONNAGE[cle]["duree_s"]
    durees = DUREES_MESUREES.get(cle)
    # Sans mesure, on retombe sur la sonde. Elle **majore**, donc ce repli est conservateur : il
    # peut écarter à tort un encodeur abordable, jamais en admettre un qui ne l'est pas.
    retenue = float(np.median(durees)) if durees else annonce
    total = retenue * K_EXTERNE * len(GERMES)
    ADMIS_PHASE_2[cle] = total <= BUDGET_PHASE_2_S
    print(f"{cle:<12} {annonce:>13.0f}s "
          f"{(f'{retenue:.0f}s' if durees else 'aucune mesure'):>18} "
          f"{(f'{annonce / retenue:.1f}×' if durees else '—'):>8} "
          f"{total / 60:>7.0f} min {'oui' if ADMIS_PHASE_2[cle] else 'NON':>9}")
print()
print(f"Plafond déclaré pour la phase 2 : {BUDGET_PHASE_2_S / 3600:.1f} h de carte par montage.")
print("La sonde majorait, et d'autant plus que le modèle est lourd à charger : ses frais fixes")
print("— chargement des poids, encodage final des 120 textes — ne s'échelonnent pas avec le")
print("nombre d'itérations, et les multiplier par le facteur les compte autant de fois.")

encodeur         sonde ×5.2   mesuré (médiane)    écart   25 corps   phase 2
e5-small                66s                64s     1.0×      27 min       oui
e5-base                116s                77s     1.5×      32 min       oui

Plafond déclaré pour la phase 2 : 1.5 h de carte par montage.
La sonde majorait, et d'autant plus que le modèle est lourd à charger : ses frais fixes
— chargement des poids, encodage final des 120 textes — ne s'échelonnent pas avec le
nombre d'itérations, et les multiplier par le facteur les compte autant de fois.


## Phase 2 — confirmation sur les 5 germes

Les finalistes seulement, et le critère appliqué **sans retouche**. Les corps déjà entraînés en
phase 1 sont relus du cache : seuls les germes non couverts sont payés.

Le choix des finalistes :

1. **les deux meilleurs taux du levier A** au pire germe ;
2. **plus le montage du premier banc** (`lr=2e-5`) s'il n'y figure pas déjà — sans quoi le banc ne
   saurait pas dire si un gain vient du taux ou de la nouvelle grille de tête à seuil réglé ;
3. **plus le meilleur encodeur du levier B**, s'il tient dans le plafond de budget mesuré
   ci-dessus. C'est la seule façon de confronter le levier B au critère : une mesure sur un germe
   ne peut, par construction, ni le franchir ni l'échouer.

In [18]:
FINALISTES = [nom for _, nom, _ in CLASSEMENT_A[:2]]
if f"e5-small@{banc.SETFIT_LR_CORPS:g}" not in FINALISTES:
    FINALISTES.append(f"e5-small@{banc.SETFIT_LR_CORPS:g}")
if ADMIS_PHASE_2.get("e5-base"):
    FINALISTES.append(nom_b)
else:
    print(f"⚠️ {nom_b} reste hors de la phase 2 : ses 25 corps dépassent le plafond déclaré.")
print("finalistes :", FINALISTES)

PHASE_2 = {}
_debut_phase_2 = time.perf_counter()
for nom in FINALISTES:
    encodeur_finaliste, lr = nom.split("@")[0], float(nom.split("@")[1])
    print(f"\n───── {nom} sur les {len(GERMES)} germes", flush=True)
    PHASE_2[nom] = montage_setfit(nom, encodeur_finaliste, lr, GERMES)
DUREE_PHASE_2 = time.perf_counter() - _debut_phase_2
print(f"\nphase 2 : {DUREE_PHASE_2 / 60:.1f} min")

finalistes : ['e5-small@2e-05', 'e5-small@0.0001', 'e5-base@2e-05']

───── e5-small@2e-05 sur les 5 germes


  e5-small@2e-05 germe 20180525 pli 0 : {'C': 10.0, 'class_weight': None} seuils (0.5, 0.3, 0.5, 0.35, 0.3, 0.4, 0.5) macro-F1 interne 0.910 — 1.8 s


  e5-small@2e-05 germe 20180525 pli 1 : {'C': 10.0, 'class_weight': None} seuils (0.45, 0.5, 0.3, 0.35, 0.5, 0.3, 0.5) macro-F1 interne 0.895 — 1.8 s


  e5-small@2e-05 germe 20180525 pli 2 : {'C': 10.0, 'class_weight': None} seuils (0.5, 0.3, 0.5, 0.35, 0.4, 0.3, 0.7) macro-F1 interne 0.871 — 1.7 s


  e5-small@2e-05 germe 20180525 pli 3 : {'C': 10.0, 'class_weight': None} seuils (0.5, 0.3, 0.45, 0.35, 0.3, 0.45, 0.5) macro-F1 interne 0.891 — 1.9 s


...

  e5-base@2e-05 germe 20221123 pli 1 : {'C': 10.0, 'class_weight': None} seuils (0.5, 0.3, 0.5, 0.3, 0.3, 0.5, 0.5) macro-F1 interne 0.948 — 1.8 s


  e5-base@2e-05 germe 20221123 pli 2 : {'C': 10.0, 'class_weight': None} seuils (0.3, 0.3, 0.5, 0.3, 0.35, 0.3, 0.5) macro-F1 interne 0.915 — 1.8 s


  e5-base@2e-05 germe 20221123 pli 3 : {'C': 10.0, 'class_weight': None} seuils (0.45, 0.3, 0.5, 0.5, 0.3, 0.3, 0.5) macro-F1 interne 0.938 — 1.8 s


  e5-base@2e-05 germe 20221123 pli 4 : {'C': 10.0, 'class_weight': None} seuils (0.5, 0.3, 0.5, 0.3, 0.3, 0.35, 0.5) macro-F1 interne 0.968 — 1.8 s



phase 2 : 69.0 min


### Le critère, appliqué

Sans retouche, et au pire des 5 germes. L'estimation ponctuelle fait foi ; l'intervalle de Wilson
est rapporté et **ne fait pas gate** — c'est la deuxième précision du critère.

In [19]:
print(f"la barre : {banc.EXACTITUDE_LEXIQUE}/{N_EXEMPLES}, à dépasser **strictement**, "
      f"au pire des {len(GERMES)} germes")
print()
print(f"{'montage':<16} {'accord exact par germe':<26} {'pire':>5} {'médiane':>8} "
      f"{'Wilson du pire':>18} {'critère':>8}")
VERDICTS = {}
for nom, enregistrements in PHASE_2.items():
    par_germe = banc.exactitudes_par_germe(enregistrements, nom, GERMES)
    v = banc.verdict_critere(par_germe)
    VERDICTS[nom] = v
    print(f"{nom:<16} {str([par_germe[g] for g in GERMES]):<26} {v['pire']:>5} "
          f"{v['mediane']:>8.0f} "
          f"[{v['wilson_pire'][0]:>5.1%} – {v['wilson_pire'][1]:>5.1%}] "
          f"{'FRANCHI' if v['franchi'] else 'non':>8}")

print()
print(f"{'lexique':<16} {'— déterministe':<26} {banc.EXACTITUDE_LEXIQUE:>5}")
print()
meilleur_nom = max(VERDICTS, key=lambda n: VERDICTS[n]["pire"])
meilleur = VERDICTS[meilleur_nom]
print(f"Le meilleur montage de la phase 2 : {meilleur_nom}, {meilleur['pire']}/{N_EXEMPLES} "
      f"au pire germe.")
print(f"Il manque {max(0, banc.EXACTITUDE_LEXIQUE + 1 - meilleur['pire'])} exemples pour franchir "
      f"la barre.")
print()
print("Ce notebook s'arrête ici : la **lecture** de ces chiffres et le verdict appartiennent à un")
print("ticket distinct, comme #52 l'a été pour le premier banc.")

la barre : 94/120, à dépasser **strictement**, au pire des 5 germes

montage          accord exact par germe      pire  médiane     Wilson du pire  critère
e5-small@2e-05   [74, 73, 72, 70, 70]          70       72 [49.4% – 66.8%]      non
e5-small@0.0001  [73, 75, 78, 73, 66]          66       73 [46.1% – 63.6%]      non
e5-base@2e-05    [72, 79, 72, 77, 77]          72       77 [51.1% – 68.3%]      non

lexique          — déterministe                94

Le meilleur montage de la phase 2 : e5-base@2e-05, 72/120 au pire germe.
Il manque 23 exemples pour franchir la barre.

Ce notebook s'arrête ici : la **lecture** de ces chiffres et le verdict appartiennent à un
ticket distinct, comme #52 l'a été pour le premier banc.


## L'artefact de prédictions

Mêmes raisons qu'au premier banc, et la même décision de
[Emplacement, outillage et reproductibilité du notebook](https://github.com/AmauryTISSOT/microservice_rgpd/issues/49) :
les prédictions par montage × germe × pli sont versionnées à part, de sorte que le ticket de
lecture puisse **recalculer le verdict hors du notebook qui l'a produit**. Chaque ligne porte en
plus l'encodeur, le `body_learning_rate` et les **seuils retenus par pli** — sans quoi le réglage
ne serait pas vérifiable.

In [20]:
CHEMIN = Path(banc.RACINE) / "exploration" / "reglage-predictions.jsonl"
TOUT = {**{f"phase1/{k}": v for k, v in PHASE_1.items()},
        **{f"phase2/{k}": v for k, v in PHASE_2.items()}}
with CHEMIN.open("w", encoding="utf-8", newline="\n") as f:
    for phase_et_nom, enregistrements in TOUT.items():
        phase = phase_et_nom.split("/")[0]
        for r in enregistrements:
            f.write(json.dumps({**r, "phase": phase}, ensure_ascii=False, sort_keys=True) + "\n")

lignes = sum(len(v) for v in TOUT.values())
print(f"{CHEMIN.relative_to(banc.RACINE)} : {lignes} lignes, {len(TOUT)} montages")

exploration\reglage-predictions.jsonl : 3120 lignes, 9 montages


## Le budget consommé, et les écarts au protocole

Le ticket demande que le budget dépensé et les configurations abandonnées soient **déclarés dans le
notebook**. Les voici, sans arrondi favorable.

In [21]:
print(f"corps contrastifs entraînés      : {BUDGET['corps_entraines']}")
print(f"corps relus du cache             : {BUDGET['corps_relus_du_cache']}")
print(f"temps d'entraînement sur la carte : {BUDGET['secondes_gpu'] / 3600:.2f} h")
print("  (le coût d'un corps est mis en cache avec lui : ce total est celui du banc, qu'il ait")
print("   été rejoué ou servi par le cache)")
print(f"phase 1 levier A                : {DUREE_PHASE_1_A / 60:.1f} min")
print(f"phase 1 levier B                : {DUREE_PHASE_1_B / 60:.1f} min")
print(f"phase 2                         : {DUREE_PHASE_2 / 60:.1f} min")
print()
print("Durées réellement observées par corps, par encodeur :")
for cle, durees in DUREES_MESUREES.items():
    print(f"  {cle:<12} médiane {np.median(durees):>5.0f} s sur {len(durees):>3} corps "
          f"(min {min(durees):.0f} s, max {max(durees):.0f} s)")
print()
print("Configurations ABANDONNÉES, et pourquoi :")
for cle in ("bge-m3", "solon-large"):
    cout = ETALONNAGE[cle]
    if "echec" in cout:
        print(f"  {cle:<12} sonde en échec : {cout['echec'][:60]}")
    else:
        print(f"  {cle:<12} sonde à ~{cout['duree_s'] * K_EXTERNE * len(GERMES) / 3600:.1f} h "
              f"pour les {K_EXTERNE * len(GERMES)} corps, VRAM de crête {cout['vram_gio']:.1f} Gio "
              f"sur une carte de 8 — jamais entraîné hors sonde, donc jamais mesuré autrement")
print(f"  {'lr hors grille':<14} la plage HPO officielle est couverte par 5 points ; rien "
      f"au-delà de 1e-3 ni en deçà de 1e-6")
print(f"  {'levier C seul':<14} mesuré à coût nul en phase 0 ; ce qu'il rapporte est intégré à "
      f"toutes les configurations des phases 1 et 2")
print(f"  {'finalistes':<14} la phase 2 ne retient que {len(FINALISTES)} montages sur "
      f"{len(PHASE_1)} mesurés en phase 1 : {FINALISTES}")

corps contrastifs entraînés      : 54
corps relus du cache             : 76
temps d'entraînement sur la carte : 2.53 h
  (le coût d'un corps est mis en cache avec lui : ce total est celui du banc, qu'il ait
   été rejoué ou servi par le cache)
phase 1 levier A                : 1.6 min
phase 1 levier B                : 6.3 min
phase 2                         : 69.0 min

Durées réellement observées par corps, par encodeur :
  e5-small     médiane    64 s sur 100 corps (min 58 s, max 97 s)
  e5-base      médiane    76 s sur  30 corps (min 72 s, max 114 s)

Configurations ABANDONNÉES, et pourquoi :
  bge-m3       sonde à ~10.0 h pour les 25 corps, VRAM de crête 11.7 Gio sur une carte de 8 — jamais entraîné hors sonde, donc jamais mesuré autrement
  solon-large  sonde à ~10.2 h pour les 25 corps, VRAM de crête 11.5 Gio sur une carte de 8 — jamais entraîné hors sonde, donc jamais mesuré autrement
  lr hors grille la plage HPO officielle est couverte par 5 points ; rien au-delà de 1e-3 ni en 

### Écarts au protocole, déclarés

1. **Le réglage de seuil de la phase 0 n'est pas parfaitement imbriqué.** Les probabilités des
   quatre plis de réglage viennent de modèles dont l'apprentissage incluait le pli tenu à l'écart.
   La fuite est de second ordre — un scalaire par étiquette sur une grille de 12 pas — et joue **en
   faveur** du réglage. Les phases 1 et 2 n'ont pas cet écart : elles règlent le seuil dans les
   plis **internes**, qui ne voient jamais le pli externe de test.
2. **La grille de tête est réduite à trois configurations dans les phases 1 et 2**, `class_weight`
   étant retiré au profit du seuil réglé. C'est la décision argumentée plus haut, pas une économie
   de budget — mais elle rend la comparaison au premier banc *non* strictement à tête constante, et
   c'est pourquoi le montage `lr=2e-5` est repassé en phase 2 : il isole ce que la nouvelle grille
   change, à taux inchangé.
3. **Le dégrossissage se lit sur deux germes**, et le levier B sur un seul en phase 1. Aucune
   conclusion n'en est tirée : seuls les chiffres de la phase 2, sur cinq germes, sont confrontés au
   critère.
4. **La sonde de budget majore, et ne décide donc rien seule.** Elle extrapole d'un facteur mesuré
   (`FACTEUR_MESURE`) et non du facteur théorique 10, mais elle compte les frais fixes d'un corps
   autant de fois que le facteur. Elle a ainsi surestimé `e5-base` d'un ordre de grandeur, et c'est
   la durée **mesurée** en phase 1 qui l'a admis en phase 2. La sonde n'a servi qu'à écarter ce que
   sa marge rendait indiscutable.
5. **La longueur de séquence est plafonnée à 128 jetons** pour tous les encodeurs. Le corpus fait
   102 caractères de médiane : rien n'est tronqué, et la comparaison porte ainsi sur l'encodeur et
   non sur la taille de sa fenêtre.
6. **Les corps sont mis en cache sur disque.** Le cache est une fonction pure de (encodeur, taux,
   germe, pli) ; le vider et relancer redonne les mêmes chiffres. Il n'est pas versionné.